In [29]:
from numba import njit, jit
import numpy as np

In [30]:
# 1. Обычная функция:
def f(xs):
    result = []
    for x in xs:
        if x > 0:
            result.append({"value": x})
        else:
            result.append(str(x))
    return result

print("type(f):", type(f))
print("f.__code__:", f.__code__)
print("f.__code__.co_varnames:", f.__code__.co_varnames)

type(f): <class 'function'>
f.__code__: <code object f at 0x111768df0, file "/var/folders/bm/j41t02r53h3b5lv3gfmlt_j40000gn/T/ipykernel_67508/1539012338.py", line 2>
f.__code__.co_varnames: ('xs', 'result', 'x')


In [31]:
# 2. Навешиваем декоратор
jf = jit(f)

print("type(jf):", type(jf))
print("jf.py_func:", jf.py_func)
print("jf.signatures:", jf.signatures)
print("jf.overloads:", jf.overloads)

type(jf): <class 'numba.core.registry.CPUDispatcher'>
jf.py_func: <function f at 0x112f15850>
jf.signatures: []
jf.overloads: OrderedDict()


In [ ]:
# 3. Первый вызов - компиляция
arg_value = 1

print("result:", jf(arg_value))
print("jf.signatures:", jf.signatures)
# print("jf.overloads keys:", list(jf.overloads.keys()))

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Invalid use of getiter with parameters (int64)

During: typing of intrinsic-call at /var/folders/bm/j41t02r53h3b5lv3gfmlt_j40000gn/T/ipykernel_67508/1539012338.py (4)

File "../../../../../../var/folders/bm/j41t02r53h3b5lv3gfmlt_j40000gn/T/ipykernel_67508/1539012338.py", line 4:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference

In [ ]:
# 4. Достаём объект результата компиляции
sig = jf.signatures[0]
cres = jf.overloads[sig]

print("type(cres):", type(cres))
print("cres.signature:", cres.signature)
print("cres.entry_point:", cres.entry_point)
print("cres.library:", cres.library)

type(cres): <class 'numba.core.compiler.CompileResult'>
cres.signature: (int64,) -> int64
cres.entry_point: <built-in method f of _dynfunc._Closure object at 0x112c40ac0>
cres.library: <Library 'f' at 0x112af2060>


In [ ]:
# 5. LLVM IR и ассемблер — это и есть низкоуровневый результат компиляции
print("\n6) LLVM IR, наша функция f:")
llvm = jf.inspect_llvm(sig)
llvm_str = "\n".join(str(llvm).splitlines()[11:107])
print("...\n" + llvm_str + "\n...")

print("\n7) Assembly, наша функция f:")
asm = jf.inspect_asm(sig)
asm_str = "\n".join(str(asm).splitlines()[4:129])
print("...\n" + asm_str + "\n...")

print("\n8) Второй вызов — уже без компиляции, используется готовая версия:")
print("result:", jf(arg_value))



6) LLVM IR, наша функция f:
...
define noundef i32 @_ZN8__main__1fB2v4B38c8tJTIeFIjxB2IKSgI4CrvQClQZ6FczSBAA_3dEx(ptr noalias nocapture writeonly initializes((0, 8)) %retptr, ptr noalias nocapture readnone %excinfo, i64 %arg.a) local_unnamed_addr #0 {
B0.endif:
  %.8944.not = icmp slt i64 %arg.a, 1
  br i1 %.8944.not, label %B84, label %B34.preheader

B34.preheader:                                    ; preds = %B0.endif
  %0 = mul i64 %arg.a, 3
  %1 = add nsw i64 %arg.a, -1
  %2 = add nsw i64 %arg.a, -2
  %3 = mul i64 %1, %2
  %4 = and i64 %3, -2
  %5 = add i64 %0, -2
  %6 = add i64 %5, %4
  br label %B84

B84:                                              ; preds = %B34.preheader, %B0.endif
  %result.2.0.lcssa = phi i64 [ 0, %B0.endif ], [ %6, %B34.preheader ]
  store i64 %result.2.0.lcssa, ptr %retptr, align 8
  ret i32 0
}

define ptr @_ZN7cpython8__main__1fB2v4B38c8tJTIeFIjxB2IKSgI4CrvQClQZ6FczSBAA_3dEx(ptr nocapture readnone %py_closure, ptr %py_args, ptr nocapture readnone %py_kw